In [1]:
#0. set up 

import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime,date,timedelta
from zoneinfo import ZoneInfo
import getpass
import timeit
from io import BytesIO
import re
from collections import namedtuple
import requests
import zipfile
from zipfile import ZipFile
import pickle
import urllib.request
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta
import random


def is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ

if is_databricks() == True:
    base = Path("/Volumes/exploration/bills_repository/files/mt_pasa/")
else:
    base = Path("C:/Users/BillNixey/OneDrive - Squadron Energy/Desktop/Working_files/New_model/mt_pasa/")

regions = ['NSW1','QLD1','SA1','TAS1','VIC1']

#
allowed_fuels = ['Wind','Battery','Solar','Black Coal','Other','Gas','Hydro','Natural Gas (Pipeline)','Diesel','Brown Coal']
now = datetime.now(ZoneInfo("Australia/Brisbane"))
s1 = str(now)
s2 = s1[:10]
s2 = s2.replace(":", "")

try:
    url2 = (
        f"https://www.neopoint.com.au/Service/Csv?"
        f"f=107%20Information%5CPlantInformation"
        f"&from={s2}%2000%3A00"
        f"&period=Daily&instances=&section=-1&key=squnix77"
    )

    df_duid = pd.read_csv(url2)
    df_duid = df_duid.rename(columns={"TransmissionLossFactor": "MLF"}) 
    df_duid = df_duid[['DUID','REGIONID','FUEL']]
    exceptions = set(df_duid['FUEL'].dropna().unique()) - set(allowed_fuels)

    if exceptions:
        print("Unexpected FUEL values found:")
        for fuel in sorted(exceptions):
            print(f"  {fuel}")
    else:
        print("All FUEL values are valid.")
    
    df_duid['FUEL'] = df_duid['FUEL'].replace('Natural Gas (Pipeline)', 'Gas')
    df_duid["FUEL"] = df_duid["FUEL"].str.replace(" ", "_", regex=False)
   
except Exception as e:
    print(f"Failed to load DUID data: {e}")


print('OK')

All FUEL values are valid.
OK


In [2]:
#1. get MT PASA DUID from Nemweb



def latest_file(url,step_back,prefix,date_len): 
    my_list = []
    req = Request(url)
    a = urlopen(req).read()
    soup = BeautifulSoup(a, 'html.parser')
    x = (soup.find_all('a')) #read all on html page
    
    #this loop finds the most recent file
    for i in x:    
        file_name = i.extract().get_text()       #take only file names
        if(prefix in file_name)==True:            
                m=len(prefix)
                date = int(file_name[m:m+date_len]) #grabs the date and converts to an integer          
                my_list.append(date) #add item to array
        else:
            pass
    #print(my_list)
    if step_back != 0:
        now = datetime.now()
        one_week_ago = now - timedelta(days=step_back)
        s1 = str(one_week_ago)
        s2 = s1[:11]
        s2 = s2.replace(":", "")
        s2 = s2.replace("-", "")
        s2 = s2.replace(" ", "")
        my_value=int(s2+"0900")
        print(my_value)
    else:
        my_value = max(my_list) #find most recent report in array 
    #print(my_list)
    location = my_list.index(my_value)    #position in array of latest date. Note 'zero' position at start of list.
    #the loop below downloads latest file a file and splits it into two df. One each for price and volume. Repeats for next most recent file.   
    my_file = x[location+len(x)-len(my_list)].extract().get_text() #This "len(x)-len(my_list)" takes into account 2 extra values in the x array
    return my_file

def get_pasa_by_duid_for_month_part_A(days):   
    print('Starting MT PASA by DUID file build part A...')
    
    url = "https://www.nemweb.com.au/Reports/CURRENT/MTPASA_DUIDAvailability/"
    prefix = 'PUBLIC_MTPASADUIDAVAILABILITY_'
        
    file = latest_file(url, days, prefix, 12)
    print(file)
        
    urllib.request.urlretrieve(url + file, base / file)
    zip_path = base / file
        # CLOSES automatically after extraction
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(base)
        # delete zip
    zip_path.unlink()
        
    csv_file = Path(file).with_suffix(".CSV")
        
    df = pd.read_csv(base / csv_file, skiprows=1,skipfooter=1,engine='python')
    zip_path2 = base / csv_file
    zip_path2.unlink()
        
    df = df[['DUID','DAY','PASAAVAILABILITY','PUBLISH_DATETIME','REGIONID']]# ,'LOAD_MAX_AVAILABILITY','GENERATION_PASA_AVAILABILITY','LOAD_PASA_AVAILABILITY'
        
    df = df.fillna(0)
    df2 =df.copy()
    
    df2 = df2.sort_values(by=["DAY",'REGIONID','DUID'])
    df2 = df2[df2.PASAAVAILABILITY > 0 ]
    
    df2 = df2.rename(columns={"DAY":"INTERVAL_DATETIME", "PASAAVAILABILITY": "GENERATION_MAX_AVAILABILITY"})
    
    
    df2 = pd.merge(df2,df_duid,how='left',on=['DUID','REGIONID'])
    df2["INTERVAL_DATETIME"] = pd.to_datetime(df2["INTERVAL_DATETIME"])
    df_sched = df2.dropna(subset=['REGIONID','FUEL'])
    
    print('earliest mt pasa day:',df_sched.INTERVAL_DATETIME.min())
    print('latest mt pasa day:',df_sched.INTERVAL_DATETIME.max())
    return df_sched
print('OK')

OK


In [3]:

#2. make one month mt pasa regional file with 7 reference years

def get_pasa_by_region_for_month(m1,y1):
    print('Starting MT PASA by region file build...')
    my_parquet = ["demand_traces_x7.parquet","ISP_solar_2026_x7.parquet","ISP_wind_2026_x7.parquet"]
    
    data = []
    
    for f in my_parquet:
        df = pd.read_parquet(base / f)
        df = df[~((df['Month'] == 2) & (df['Day'] == 29))] #remove leap years
        df = df.copy()
        df = df[(df['Month'] == m1)] 
        df['Year'] = y1
        if f == "demand_traces_x7.parquet":         
            pass
        else:
            df['Hour'] = df['INTERVAL_DATETIME'].dt.hour
            df['Minute'] = df['INTERVAL_DATETIME'].dt.minute
            
        df['INTERVAL_DATETIME'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])
        df = df.drop(columns=['Hour', 'Minute','Year', 'Month', 'Day']) 
       
        data.append(df)
    
    df_d,df_s,df_w = data[0],data[1],data[2]
    
    df_s_pasa = (df_s.groupby(['INTERVAL_DATETIME', 'REF_YEAR', 'REGIONID'],as_index=False)['MW'].sum().rename(columns={'MW': 'SS_SOLAR_UIGF'}))
    df_w_pasa = (df_w.groupby(['INTERVAL_DATETIME', 'REF_YEAR', 'REGIONID'],as_index=False)['MW'].sum().rename(columns={'MW': 'SS_WIND_UIGF'}))
    df_pasa = pd.merge(df_d, df_s_pasa, on=['INTERVAL_DATETIME', 'REF_YEAR', 'REGIONID'], how = 'left')
    df_pasa = pd.merge(df_pasa, df_w_pasa, on=['INTERVAL_DATETIME', 'REF_YEAR', 'REGIONID'], how = 'left')
    df_pasa = df_pasa.fillna(0)
    
    df_pasa = (df_pasa.sort_values(['REF_YEAR','INTERVAL_DATETIME','REGIONID']).reset_index(drop=True))

    return df_s,df_w, df_pasa

print('OK')

OK


In [4]:
#3.#get random coal units based on recent months 
#make month ahead duid file
# add wind and solar for each ref year

def get_online_coal_units(m1,df_pasa_duid):
    coal_duids = {'NSW1': ['BW01','BW02','BW03','BW04','VP5','VP6','MP1','MP2','ER01','ER02','ER03','ER04'],\
                      'VIC1': ['LYA1','LYA2','LYA3','LYA4','LOYYB1','LOYYB2','YWPS1','YWPS2','YWPS3','YWPS4'],\
                      'QLD1': ['CPP_3','CPP_4','CALL_A_1','CALL_A_2','CALL_A_3','CALL_B_1','CALL_B_2','GSTONE1','GSTONE2','GSTONE3','GSTONE4',\
                               'GSTONE5','GSTONE6','KPP_1',\
                               'MPP_1','MPP_2','STAN-1','STAN-2','STAN-3','STAN-4','SWAN_B_1','SWAN_B_3','TARONG#1','TARONG#2','TARONG#3','TARONG#4',\
                               'TNPS1']}
    # this was determined from code furthe below commented out
    
    monthly_count_of_coal_units = {
    1:  {"VIC1": 8.13, "NSW1": 10.68, "QLD1": 18.61},
    2:  {"VIC1": 8.07, "NSW1": 11.00, "QLD1": 18.82},
    3:  {"VIC1": 9.10, "NSW1": 11.42, "QLD1": 19.13},
    4:  {"VIC1": 9.13, "NSW1": 10.93, "QLD1": 18.50},
    5:  {"VIC1": 9.10, "NSW1": 10.94, "QLD1": 16.16},
    6:  {"VIC1": 9.50, "NSW1": 10.47, "QLD1": 16.13},
    7:  {"VIC1": 9.90, "NSW1": 11.32, "QLD1": 18.26},
    8:  {"VIC1": 8.65, "NSW1": 10.23, "QLD1": 19.26},
    9:  {"VIC1": 6.87, "NSW1": 9.07,  "QLD1": 18.30},
    10: {"VIC1": 6.77, "NSW1": 8.26,  "QLD1": 17.94},
    11: {"VIC1": 7.57, "NSW1": 8.60,  "QLD1": 18.47},
    12: {"VIC1": 7.55, "NSW1": 9.35,  "QLD1": 18.84},
    }
    
    surplus_list = []
    for r in ["NSW1","VIC1","QLD1"]:
        target = monthly_count_of_coal_units[m1][r]
        df = df_pasa_duid[(df_pasa_duid.FUEL.isin(['Brown_Coal','Black_Coal']))& (df_pasa_duid.REGIONID == r)]
        avg_units = (df.groupby(df["INTERVAL_DATETIME"].dt.date)["DUID"].nunique().mean())
    
        surplus_units = round(abs(avg_units - target))
        print("surplus units",r,surplus_units)
    
        selected = random.sample(coal_duids[r], surplus_units)
        surplus_list += selected    
    print(surplus_list)
   
    return surplus_list



def get_pasa_by_duid_for_month_part_B(df_s,df_w,df_sched,m1,y1):
    print('Starting MT PASA by DUID file build part B...')
    data2 = []
    for d in [df_s,df_w,df_sched]:
        df = d
        df = df[(df['INTERVAL_DATETIME'].dt.year == y1 ) & (df['INTERVAL_DATETIME'].dt.month == m1) ]
        data2.append(df)
    
    df_s2,df_w2,df_d2 = data2[0],data2[1],data2[2]
    
    df_w2 = df_w2.copy() 
    df_s2 = df_s2.copy()
    
    df_w2['FUEL'] = 'Wind'
    df_s2['FUEL'] = 'Solar'
    
    df_ss = pd.concat([df_w2,df_s2])
    df_ss = df_ss.rename(columns={"MW": "GENERATION_MAX_AVAILABILITY"})
    
    df_d2 = df_d2.drop(columns=['PUBLISH_DATETIME'])
    
    df_d3 = df_d2.loc[df_d2.index.repeat(48)].copy()
    
    # Add 0, 30, 60, ..., 1410 minutes within each original row
    df_d3["INTERVAL_DATETIME"] = (df_d3["INTERVAL_DATETIME"]+ pd.to_timedelta(np.tile(np.arange(48) * 30, len(df_d2)), unit="min"))
    
    df_d3 = df_d3.reset_index(drop=True)
    
    df_d3['REF_YEAR'] = 0
    
    df_out = pd.concat([df_ss,df_d3])
    
    df_out=df_out.sort_values(by=["INTERVAL_DATETIME","REGIONID",'DUID'],ascending=True).reset_index(drop=True)
    
    df_out=df_out[df_out.GENERATION_MAX_AVAILABILITY >0]
    #remove extra coal units
    excluded_list = get_online_coal_units(m1,df_out)
    
    df_out = df_out[~df_out["DUID"].isin(excluded_list)]
    
    return df_out
print('OK')

OK


In [9]:
#4. run model and prepare PASA files for given months
print("Starting...")

days= 45 #zero for latest file. need a count back if doing a backcast
df_sched = get_pasa_by_duid_for_month_part_A(days)

my_date = date(2026, 7, 1)
months_to_solve = 1

month_array1 = []
month_array2 = []
for _ in range(months_to_solve):
    m1 = my_date.month
    y1 = my_date.year
    
    print("Solving:",m1, y1)

    df_s,df_w,df_pasa = get_pasa_by_region_for_month(m1,y1)
    month_array1.append(df_pasa)

    df_out = get_pasa_by_duid_for_month_part_B(df_s,df_w,df_sched,m1,y1)
    month_array2.append(df_out)
    
    my_date += relativedelta(months=1)
    
   
df_final1 = pd.concat(month_array1)
df_final2 = pd.concat(month_array2)

print("Saving file MTPASA_by_ref_years.parquet")
df_final1.to_parquet(base /'MTPASA_by_ref_years.parquet',index=False)
print('FINAL: earliest mt pasa region day:',df_final1.INTERVAL_DATETIME.min())
print('FINAL: latest mt pasa region day:',df_final1.INTERVAL_DATETIME.max())

print("Saving file MTPASA_duid_daily.parquet")
df_final2.to_parquet(base /"MTPASA_duid_daily.parquet", index=False)
print('FINAL: earliest mt pasa duid day:',df_final2.INTERVAL_DATETIME.min())
print('FINAL: latest mt pasa duid day:',df_final2.INTERVAL_DATETIME.max())
print('Finished')

Starting...
Starting MT PASA by DUID file build part A...
202606230900
PUBLIC_MTPASADUIDAVAILABILITY_202606230900_0000000523843808.zip
earliest mt pasa day: 2026-06-28 00:00:00
latest mt pasa day: 2029-06-24 00:00:00
Solving: 7 2026
Starting MT PASA by region file build...
Starting MT PASA by DUID file build part B...
surplus units NSW1 0
surplus units VIC1 0
surplus units QLD1 2
['GSTONE2', 'GSTONE4']
Saving file MTPASA_by_ref_years.parquet
FINAL: earliest mt pasa region day: 2026-07-01 00:00:00
FINAL: latest mt pasa region day: 2026-07-31 23:30:00
Saving file MTPASA_duid_daily.parquet
FINAL: earliest mt pasa duid day: 2026-07-01 00:00:00
FINAL: latest mt pasa duid day: 2026-07-31 23:30:00
Finished


In [34]:

# this script gets the average count of coal units online by region
import pandas as pd

regions = ['VIC1','NSW1','QLD1']

coal_duids = {'NSW1':['BW01','BW02','BW03','BW04','VP5','VP6','MP1','MP2','ER01','ER02','ER03','ER04'],'VIC1':['LYA1','LYA2','LYA3','LYA4','LOYYB1','LOYYB2','YWPS1','YWPS2','YWPS3','YWPS4'],'QLD1':['CPP_3',	'CPP_4',	'CALL_A_1',	'CALL_A_2',	'CALL_A_3',	'CALL_B_1',	'CALL_B_2',	'GSTONE1',	'GSTONE2',	'GSTONE3',	'GSTONE4',	'GSTONE5',	'GSTONE6',	'KPP_1',	'MPP_1',	'MPP_2',	'STAN-1',	'STAN-2',	'STAN-3',	'STAN-4',	'SWAN_B_1',	'SWAN_B_3',	'TARONG#1',	'TARONG#2',	'TARONG#3',	'TARONG#4',	'TNPS1']}

my_list = []
for r in regions:


    url2 = (f"https://www.neopoint.com.au/Service/Csv?f=103%20Generation%20and%20Load%5CRegion%20%2F%20Plant%20Energy%20MWh%20Actual%20Generation%20Daily&from=2025-08-01%2000%3A00&period=Yearly&instances=GEN%3B{r}&section=-1&key=squnix77")
    #url2 = (f"https://www.neopoint.com.au/Service/Csv?f=103%20Generation%20and%20Load%5CRegion%20TotalCleared%20per%20DUID%205min%20(stack)&from=2025-10-01%2000%3A00&period=Monthly&instances=GEN%3B{r}&section=-1&key=squnix77")
    df_data = pd.read_csv(url2)

    df_data.columns = df_data.columns.str.replace('.MWh_GEN', '', regex=False)
    df_data = df_data.loc[:, df_data.columns.isin(['DateTime'] + coal_duids[r])]
    duid_cols = df_data.columns.drop('DateTime')
    df_data[r] = df_data[duid_cols].gt(1000).sum(axis=1) #has to be generating more than 1 GWh per day
    
    df_data2 = df_data[['DateTime',r]]
    my_list.append(df_data)

df_final = pd.concat([df.set_index('DateTime') for df in my_list],axis=1).reset_index()

df_final['DateTime'] = pd.to_datetime(df_final['DateTime'])

df_final['month'] = df_final['DateTime'].dt.month

df_monthly = (df_final.groupby('month', as_index=False)[regions].mean())

df_monthly[regions] = df_monthly[regions].round(2)

df_monthly


,month,VIC1,NSW1,QLD1
0,1,8.13,10.68,18.61
1,2,8.07,11.00,18.82
2,3,9.10,11.42,19.13
3,4,9.13,10.93,18.50
4,5,9.10,10.94,16.16
5,6,9.50,10.47,16.13
6,7,9.90,11.32,18.26
7,8,8.65,10.23,19.26
8,9,6.87,9.07,18.30
9,10,6.77,8.26,17.94


In [29]:

#df_data['DateTime'] = pd.to_datetime(df_data['DateTime'])
#df_data[df_data.DateTime.dt.day == 16]

,DateTime,LOYYB1,LOYYB2,LYA1,LYA2,LYA3,LYA4,YWPS1,YWPS2,YWPS3,YWPS4,VIC1
15,2025-08-16,12604.06,12259.66,11638.58,11340.50,0.00,11484.08,0.00,7737.00,7925.63,8246.13,8
46,2025-09-16,13018.19,0.00,8466.08,8530.75,8584.08,8596.00,0.00,0.00,8874.73,0.00,6
76,2025-10-16,0.00,11353.06,9595.75,0.00,9869.17,0.00,7782.38,0.00,6442.38,4175.39,6
107,2025-11-16,8512.53,8476.16,9338.33,0.00,8952.00,8630.50,7909.33,0.00,7643.44,5629.90,8
137,2025-12-16,11748.13,11847.56,11194.08,0.00,11140.67,11199.58,7669.04,0.00,8246.25,0.00,7
168,2026-01-16,7793.72,7887.69,7212.50,0.00,0.00,7401.92,7917.67,0.00,0.00,7203.87,6
199,2026-02-16,12842.65,12564.98,10428.00,11514.67,10291.83,10398.25,0.00,2299.42,8143.86,8470.45,9
227,2026-03-16,13658.91,13812.31,13113.83,4403.67,13083.17,13033.50,7917.50,7873.56,8279.53,8646.35,10
258,2026-04-16,10502.03,10507.47,9040.25,10704.92,9780.83,9033.50,7921.29,7681.27,23.41,7993.95,10
288,2026-05-16,11129.44,11084.53,0.00,12296.75,11715.75,4458.67,3.81,7677.94,7764.45,7759.17,9


In [16]:
#base = Path("C:/Users/BillNixey/OneDrive - Squadron Energy/Desktop/Working_files/New_model/mt_pasa/")
#df_final.to_csv(base / "test.csv")

In [10]:
'''
monthly_avg = {
    1:  {"VIC1": 8.19, "NSW1": 10.74, "QLD1": 18.68},
    2:  {"VIC1": 8.07, "NSW1": 11.07, "QLD1": 19.21},
    3:  {"VIC1": 9.29, "NSW1": 11.42, "QLD1": 19.45},
    4:  {"VIC1": 9.87, "NSW1": 11.00, "QLD1": 18.67},
    5:  {"VIC1": 9.32, "NSW1": 11.00, "QLD1": 16.35},
    6:  {"VIC1": 9.60, "NSW1": 10.50, "QLD1": 16.27},
    7:  {"VIC1": 9.94, "NSW1": 11.39, "QLD1": 18.29},
    8:  {"VIC1": 8.84, "NSW1": 10.42, "QLD1": 19.52},
    9:  {"VIC1": 7.00, "NSW1": 9.10,  "QLD1": 18.97},
    10: {"VIC1": 6.77, "NSW1": 8.39,  "QLD1": 19.06},
    11: {"VIC1": 7.63, "NSW1": 8.63,  "QLD1": 19.33},
    12: {"VIC1": 7.65, "NSW1": 9.35,  "QLD1": 18.87},
}

coal_duids = {'NSW1': ['BW01','BW02','BW03','BW04','VP5','VP6','MP1','MP2','ER01','ER02','ER03','ER04'],\
                      'VIC1': ['LYA1','LYA2','LYA3','LYA4','LOYYB1','LOYYB2','YWPS1','YWPS2','YWPS3','YWPS4'],\
                      'QLD1': ['CPP_3','CPP_4','CALL_A_1','CALL_A_2','CALL_A_3','CALL_B_1','CALL_B_2','GSTONE1','GSTONE2','GSTONE3','GSTONE4',\
                               'GSTONE5','GSTONE6','KPP_1',\
                               'MPP_1','MPP_2','STAN-1','STAN-2','STAN-3','STAN-4','SWAN_B_1','SWAN_B_3','TARONG#1','TARONG#2','TARONG#3','TARONG#4',\
                               'TNPS1']}

surplus_list = []
for r in ["NSW1","VIC1","QLD1"]:
    target = monthly_avg[m1][r]
    df = df_final2[(df_final2.FUEL.isin(['Brown_Coal','Black_Coal']))& (df_final2.REGIONID == r)]
    avg_units = (df.groupby(df["INTERVAL_DATETIME"].dt.date)["DUID"].nunique().mean())

    surplus_units = round(abs(avg_units - target))
    print("surplus units",r,surplus_units)

    selected = random.sample(coal_duids[r], surplus_units)
    surplus_list += selected    
print(surplus_list)
'''


surplus units NSW1 0
surplus units VIC1 0
surplus units QLD1 0
[]
